In [1]:
import json
import datetime as dt
import pandas as pd

from dateutil.relativedelta import relativedelta

from rockyclickup.wrapper import Session as rcu_session
from rockyclickup.utils import response_to_dataframe as rcu_res_to_df

from rockyelevate.wrapper import Session as elv_session
from rockyelevate.utils import response_to_dataframe as elv_res_to_df

In [2]:
elv = elv_session("PROD", multithread=True, max_threads=40) 

In [3]:
# today = dt.datetime.now()
today = dt.datetime(2026, 1, 29)
month_to_roll = dt.datetime(2026, 3, 1)

In [4]:
plans_to_roll = pd.read_pickle(f"{month_to_roll.strftime("%B%y").lower()}_plans_to_roll.pkl")
print(f"{len(plans_to_roll)} {month_to_roll.strftime("%B").lower()} plans to roll")

14 march plans to roll


In [5]:
def build_new_plan_year_bodies(plans_to_roll):
    plan_year_df = plans_to_roll[['organization_id', "plan_year.id", "plan_year.valid_from", "plan_year.valid_to"]].drop_duplicates()

    new_plan_year_bodies = []
    for _, row in plan_year_df.iterrows():
        new_valid_from = row['plan_year.valid_to'] + relativedelta(days=1)
        
        # make sure new valid from is the first day of the month
        if new_valid_from.day != 1:
            raise ValueError(f"{new_valid_from.strftime("%m/%d/%Y")} is not the first day of the month ({row['organization_id']})")
        
        new_valid_to = (new_valid_from + relativedelta(years=1)) - relativedelta(days=1)

        # make sure new valid to is the last day of the month
        if (new_valid_to + relativedelta(days=1)).day != 1:
            raise ValueError(f"{new_valid_to.strftime("%m/%d/%Y")} is not the last day of the month ({row['organization_id']})")
        
        valid_from_str = new_valid_from.strftime("%m/%d/%Y")
        valid_to_str = new_valid_to.strftime("%m/%d/%Y")

        new_plan_year_bodies.append({
            "organization_id": row['organization_id'],
            "name": f"{valid_from_str} - {valid_to_str}",
            "valid_from": valid_from_str,
            "valid_to": valid_to_str,
            "prior_plan_year_id": row['plan_year.id']
        })

    return new_plan_year_bodies


In [6]:
new_plan_year_bodies = build_new_plan_year_bodies(plans_to_roll)
new_plan_year_df = elv_res_to_df(new_plan_year_bodies)
exisiting_plan_years = elv.get_plan_years(oids=new_plan_year_df['organization_id'].to_list())
existing_plan_year_df = elv_res_to_df(exisiting_plan_years)

for col in ['valid_from', "valid_to"]:
    existing_plan_year_df[col] = pd.to_datetime(existing_plan_year_df[col])


In [8]:
def find_existing_plan_years(needed_plan_years, existing_plan_years):
    df = needed_plan_years.copy()
    df['id'] = None

    for index, row in df.iterrows():
        org_pys = existing_plan_years[existing_plan_years['organization_id'] == row['organization_id']]
        
        matching_name = org_pys[org_pys['name'] == row['name']]
        if not matching_name.empty:
            df.loc[index, 'id'] = matching_name.iloc[0]['id']
            continue

        matching_dates = org_pys[
            (org_pys['valid_from'] == row['valid_from']) &
            (org_pys['valid_to'] == row['valid_to'])
        ]
        if not matching_dates.empty:
            df.loc[index, 'id'] = matching_name.iloc[0]['id']
            continue

    return df


In [14]:
match_plan_years_df = find_existing_plan_years(new_plan_year_df, existing_plan_year_df)
plan_years_to_create = match_plan_years_df[match_plan_years_df['id'].isna()].drop(columns="id")
plan_year_bodies_to_send = [row.to_dict() for _, row in  plan_years_to_create.iterrows()]

In [ ]:
def create_plan_years(plan_year_bodies):
    if not plan_year_bodies:
        return []

    responses = []
    for pyb in plan_year_bodies:
        elv_res = elv.post(f"{elv.basepath}/plan-years", payload=pyb)

        responses.append((pyb, elv_res))

    return responses        